<a href="https://colab.research.google.com/github/tuonggb7734-pixel/Data-Processing-Programming-lab/blob/main/lab_04.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 4 · Lập hồ sơ một quận bằng pandas

**Lập trình xử lý dữ liệu (LTXLDL) · 2627-1 · Giờ thực hành · bài 4**

> 💡 File → **Save a copy in Drive** trước khi sửa.

Notebook bài giảng Bài 4 khám phá toàn thành phố. Trong lab này, bạn lập **hồ sơ chi tiết
của một quận** — Ñuñoa — và so sánh quận này với mặt bằng chung của Santiago.

## Cách làm việc trong bài lab

- Bài tập được chia thành từng bước; mỗi bước có ô `TODO` và phần kiểm tra `assert`.
  Hoàn thành toàn bộ `assert` nghĩa là kết quả đáp ứng yêu cầu.
- Phần khởi động và bài có hướng dẫn: bạn nên **tự gõ, không dùng AI** — các bài kiểm tra
  trên lớp (🚫 đóng) đánh giá các kỹ năng này.
- Bài tự làm ở cuối được gắn nhãn ✅ mở: bạn được dùng AI, kèm trách nhiệm khai báo
  theo chính sách AI của môn.
- Nếu chưa giải quyết được một bước sau 3 phút, hãy gọi giảng viên thực hành đến hỗ trợ.

## Mục tiêu

Sau bài lab, bạn sẽ:

1. Chạy thói quen 5 bước và **rút ra số liệu** từ output, không chỉ thực thi lệnh.
2. Phân biệt `df["cột"]`, `df[["a", "b"]]`, `df[mask]` và `df.loc[mask, "cột"]`; lọc bool nhiều điều kiện; `nsmallest`.
3. Sửa bảng bằng `.loc[hàng, cột]` và **tính lại cột phụ thuộc** sau khi sửa: cột lưu giá trị đã tính, không lưu công thức.
4. Tính tỷ lệ bằng mean-trên-bool và nói rõ **mẫu số**: NaN ảnh hưởng kết quả thế nào.
5. Xuất một bảng hồ sơ ra file CSV, đọc lại và kiểm một bất biến.


## Phần 0 · Khởi động (~10 phút)

In [ ]:
import pandas as pd

URL = ("https://data.insideairbnb.com/chile/rm/santiago/"
       "2026-06-29/visualisations/listings.csv")
df = pd.read_csv(URL)
df.shape
df.columns

Index(['id', 'name', 'host_id', 'host_profile_id', 'host_name',
       'neighbourhood_group', 'neighbourhood', 'latitude', 'longitude',
       'room_type', 'price', 'minimum_nights', 'number_of_reviews',
       'last_review', 'reviews_per_month', 'calculated_host_listings_count',
       'availability_365', 'number_of_reviews_ltm', 'license'],
      dtype='object')

In [ ]:
# W1 — chọn cột: một tên -> Series, danh sách -> DataFrame
# TODO: lấy cột price thành Series, và bảng con gồm 2 cột name, price
cot_gia = df["price"]
bang_con = df[["name", "price"]]

# --- Ô kiểm tra ---
assert type(cot_gia).__name__ == "Series"
assert type(bang_con).__name__ == "DataFrame" and list(bang_con.columns) == ["name", "price"]
print("W1 ổn — nhớ 2 lớp ngoặc cho DataFrame.")

W1 ổn — nhớ 2 lớp ngoặc cho DataFrame.


In [ ]:
# W2 — mean trên bool trả về tỷ lệ (quy tắc bài 3, áp dụng trên pandas)
# TODO: tỷ lệ chỗ ở loại "Private room" trong toàn bảng
ty_le_private = (df["room_type"] == "Private room").mean()

# --- Ô kiểm tra ---
assert round(ty_le_private, 3) == 0.184
print(f"W2 ổn: {ty_le_private:.1%} là phòng riêng")

W2 ổn: 18.4% là phòng riêng


In [ ]:
# W3 — [] hay .loc? Ngoặc vuông nhận tên cột hoặc một mask; .loc nhận cả hàng lẫn cột
# TODO: (a) các dòng nguyên căn bằng df[mask]; (b) chỉ cột price của các dòng đó bằng df.loc[mask, "price"]
mask_nc = df["room_type"] == "Entire home/apt"
nguyen_can = df[mask_nc]
gia_nguyen_can = df.loc[mask_nc, "price"]

# --- Ô kiểm tra ---
assert type(mask_nc).__name__ == "Series" and mask_nc.dtype == bool
assert type(nguyen_can).__name__ == "DataFrame" and len(nguyen_can) == 15004
assert type(gia_nguyen_can).__name__ == "Series" and len(gia_nguyen_can) == 15004
assert gia_nguyen_can.median() == 64900.0
print("W3 ổn — df[mask] cho hàng, df.loc[mask, cột] cho đúng ô cần lấy.")


W3 ổn — df[mask] cho hàng, df.loc[mask, cột] cho đúng ô cần lấy.


## Phần 1 · Hồ sơ quận Ñuñoa (~65 phút)

Tình huống: một nhà đầu tư quan tâm **quận Ñuñoa** và cần bản hồ sơ một trang:
quy mô, mức giá so với thành phố, cơ cấu loại phòng và mức thiếu dữ liệu của quận.

### Bước 1 · Thói quen 5 bước — chạy và đọc kết quả

Chạy 5 lệnh dưới, rồi **điền các con số đọc được** vào ô kiểm tra bên dưới
(sao chép giá trị từ output, không ước đoán).

In [ ]:
df.shape

(18534, 19)

In [ ]:
df.sample(3, random_state=1)[["name", "neighbourhood", "room_type", "price"]] # .sample(3): chọn ngẫu nhiên 3 hàng từ bảng
                                                                              # random_state=1: seed

,name,neighbourhood,room_type,price
4714,Depto Las Condes-Estadio Español,Las Condes,Entire home/apt,85588.0
2621,Habitación Matrimonial + cama 1 plaza,Providencia,Private room,30331.0
8659,"Centro De Santiago, Lastrarías",Santiago,Entire home/apt,46218.0


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18534 entries, 0 to 18533
Data columns (total 19 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   id                              18534 non-null  int64  
 1   name                            18534 non-null  object 
 2   host_id                         18534 non-null  int64  
 3   host_profile_id                 18534 non-null  int64  
 4   host_name                       18534 non-null  object 
 5   neighbourhood_group             0 non-null      float64
 6   neighbourhood                   18534 non-null  object 
 7   latitude                        18534 non-null  float64
 8   longitude                       18534 non-null  float64
 9   room_type                       18534 non-null  object 
 10  price                           17688 non-null  float64
 11  minimum_nights                  18528 non-null  float64
 12  number_of_reviews               

In [ ]:
df["price"].describe().round(0)

,price
count,17688.0
mean,118200.0
std,1226089.0
min,979.0
25%,41500.0
50%,59000.0
75%,90814.0
max,97000045.0


In [ ]:
df["room_type"].value_counts()

,count
room_type,
Entire home/apt,15004
Private room,3413
Shared room,61
Hotel room,56


In [ ]:
# TODO: điền 4 con số vừa đọc được từ 5 output trên
so_dong = 18534            # từ shape
so_cot = 19             # từ shape
so_thieu_price = 846      # từ info(): 18534 - (số non-null của price)
so_loai_phong = 4      # từ value_counts(): có mấy loại phòng?

# --- Ô kiểm tra ---
assert so_dong == 18534 and so_cot == 19
assert so_thieu_price == 846 and so_loai_phong == 4
print("Đọc output chuẩn — kỹ năng bị xem nhẹ nhất và cần nhất.")

Đọc output chuẩn — kỹ năng bị xem nhẹ nhất và cần nhất.


### Bước 2 · Cắt riêng quận Ñuñoa

In [ ]:
# TODO: lọc các dòng có neighbourhood == "Ñuñoa" (dùng đúng tên xuất hiện trong output)
nu = df[df["neighbourhood"] == "Ñuñoa"]
# TODO: Ñuñoa chiếm bao nhiêu phần của thành phố?
ty_trong = len(nu) / len(df)

# --- Ô kiểm tra ---
assert len(nu) == 1813 and round(ty_trong, 3) == 0.098
print(f"Ñuñoa: {len(nu):,} chỗ ở = {ty_trong:.1%} thị trường.")

Ñuñoa: 1,813 chỗ ở = 9.8% thị trường.


### Bước 3 · Lọc kết hợp trong quận

Nhà đầu tư hỏi: *"trong Ñuñoa, có bao nhiêu căn **nguyên căn** giá **từ 60.000 CLP trở
xuống**?"* — phân khúc họ muốn cạnh tranh.

In [ ]:
# TODO: đếm số dòng của nu thoả cả hai điều kiện (nhớ ngoặc quanh từng vế + &)
so_can = ((nu["room_type"] == "Entire home/apt") & (nu["price"] <= 60000)).sum()

# --- Ô kiểm tra ---
assert so_can == 539
print(f"{so_can} căn — phân khúc khá đông.")

539 căn — phân khúc khá đông.


### Bước 4 · Kiểm tra 5 chỗ ở có giá thấp nhất quận

In [ ]:
# TODO: dùng nsmallest lấy 5 dòng giá thấp nhất của nu, chỉ giữ 2 cột name và price
re_nhat = nu.nsmallest(5, "price")[["name", "price"]]

# --- Ô kiểm tra ---
assert re_nhat.shape == (5, 2) and re_nhat["price"].min() == 2228.0
re_nhat

,name,price
4827,Dormitorio + baño privado ÑUÑOA,2228.0
16514,Confortables habitaciones en Barrio Italia.,3958.0
16525,Moderno apartamento equipado cercano a Metro/Mall,5714.0
4821,"Gran Departamento AMOBLADO, Ñuñoa, metro Grecia",6073.0
4231,Nice and cosy bedroom,6501.0


2.228 CLP/đêm (~60 nghìn đồng) cho một "dormitorio" là mức giá cần được kiểm tra.
Hãy ghi nhận trường hợp này; bài 10 sẽ trình bày quy tắc gắn cờ các giá trị tương tự.

### Bước 5 · Sửa bảng bằng `.loc` và tính lại cột phụ thuộc

Nhà đầu tư muốn xem báo giá **3 đêm** cho Ñuñoa nếu **giảm 10% giá các chỗ ở nguyên căn**.
Làm trên một bản sao để không đụng `nu`. Trước khi chạy ô kiểm tra thứ hai, dự đoán: cột `tien_3_dem` có tự đổi theo giá mới không?


In [ ]:
bao_gia = nu[["id", "room_type", "price"]].copy()
bao_gia["tien_3_dem"] = bao_gia["price"] * 3

# TODO: giảm 10% giá cho các dòng nguyên căn, gán thẳng vào bảng bằng .loc[mask, "price"]
mask_nc_nu = bao_gia["room_type"] == "Entire home/apt"
bao_gia.loc[mask_nc_nu, "price"] *= 0.9

# --- Ô kiểm tra ---
assert mask_nc_nu.sum() == 1457
assert round(bao_gia["price"].median(), 1) == 55050.3
assert nu["price"].median() == 60981.0, "nu không được đổi — bạn đã sửa bản sao chưa?"
print("Giá đã sửa trên bản sao; nu giữ nguyên.")


Giá đã sửa trên bản sao; nu giữ nguyên.


In [ ]:
# tien_3_dem vẫn là con số tính từ giá CŨ: cột lưu giá trị, không lưu công thức
assert bao_gia["tien_3_dem"].median() == 182943.0

# TODO: tính lại tien_3_dem từ giá mới
bao_gia["tien_3_dem"] = bao_gia["price"] * 3

# --- Ô kiểm tra ---
assert round(bao_gia["tien_3_dem"].median(), 1) == 165150.9
print("Đã tính lại — trung vị báo giá 3 đêm: 165.151 CLP.")


Đã tính lại — trung vị báo giá 3 đêm: 165.151 CLP.


Nếu bạn viết `bao_gia["price"][mask_nc_nu] = ...` (hai lần ngoặc), pandas 3 báo `ChainedAssignmentError` và **bảng không đổi**: ngoặc thứ nhất lấy ra một Series, ngoặc thứ hai ghi vào Series đó. Dạng viết này gặp nhiều trong code cũ và code do AI sinh ra; xem lại slide "Hai lần ngoặc thì bảng không đổi" của Bài 4.


### Bước 6 · So sánh Ñuñoa với thành phố và xem ảnh hưởng của NaN

So từng chỗ ở của Ñuñoa với **trung vị toàn thành phố** (59.000 CLP).

In [ ]:
med_tp = df["price"].median()

# TODO: tính tỷ lệ chỗ ở Ñuñoa có giá THẤP HƠN med_tp, theo 2 mẫu số:
ty_le_tho = (nu["price"] < med_tp).mean()        # cách 1: mean trực tiếp trên (nu["price"] < med_tp)
ty_le_sach = (nu.loc[nu["price"].notna(), "price"] < med_tp).mean()       # cách 2: chỉ tính trên các hàng có giá: nu.loc[nu["price"].notna(), "price"]

# --- Ô kiểm tra ---
assert round(ty_le_tho, 3) == 0.444
assert round(ty_le_sach, 3) == 0.467
print(f"Thô: {ty_le_tho:.1%} — Sạch: {ty_le_sach:.1%}. Mẫu số đổi thì tỷ lệ đổi: lệch 2.3 điểm %.")

Thô: 44.4% — Sạch: 46.7%. Mẫu số đổi thì tỷ lệ đổi: lệch 2.3 điểm %.


Vì sao lệch? Phép so `NaN < 59000` trả `False`, nên 90 chỗ ở không khai giá bị đếm
như thể "không rẻ hơn", làm mẫu số tăng thêm 90. Cách 2 xác định tập hàng trước (mẫu số = các chỗ ở có giá) rồi mới tính.
Bạn cần chọn cách tính phù hợp và nói rõ rằng tỷ lệ được tính trên các chỗ ở có giá. Đây là
bài học `if g is not None` của lab 2, phiên bản pandas.

### Bước 7 · Cơ cấu loại phòng và mức thiếu — hoàn thiện hồ sơ

In [ ]:
# TODO: cơ cấu loại phòng của Ñuñoa theo TỶ LỆ (value_counts với normalize=True)
co_cau_nu = nu["room_type"].value_counts(normalize=True)
# TODO: tỷ lệ thiếu giá của Ñuñoa (%, làm tròn 2 chữ số)
thieu_nu = round(nu["price"].isna().mean() * 100, 2)

# --- Ô kiểm tra ---
assert round(co_cau_nu["Entire home/apt"], 3) == 0.804
assert thieu_nu == 4.96
print(f"Nguyên căn {co_cau_nu['Entire home/apt']:.1%} · thiếu giá {thieu_nu}% (toàn TP: 4.56%)")

Nguyên căn 80.4% · thiếu giá 4.96% (toàn TP: 4.56%)


### Bước 8 · Xuất hồ sơ ra file

In [ ]:
ho_so = pd.DataFrame([{
    "quan": "Ñuñoa",
    "so_cho_o": len(nu),
    "ty_trong_thi_truong": round(ty_trong, 3),
    "gia_trung_vi": nu["price"].median(),
    "ty_le_duoi_tv_thanh_pho": round(ty_le_sach, 3),
    "ty_le_thieu_gia_pct": thieu_nu,
}])
ho_so.to_csv("ho_so_nunoa.csv", index=False)

# TODO: đọc lại file vừa ghi để kiểm tra (bất biến: cùng shape, cùng số chỗ ở, cùng trung vị)
doc_lai = pd.read_csv("ho_so_nunoa.csv")

# --- Ô kiểm tra ---
assert doc_lai.shape == (1, 6) and doc_lai.loc[0, "so_cho_o"] == 1813
assert doc_lai.loc[0, "gia_trung_vi"] == 60981.0
print("Hồ sơ đã lưu — trung vị Ñuñoa 60.981 CLP, nhỉnh hơn thành phố 3%.")

Hồ sơ đã lưu — trung vị Ñuñoa 60.981 CLP, nhỉnh hơn thành phố 3%.


## Phần 2 · Bài tự làm ✅ mở (làm sớm tại lớp hoặc làm tại nhà)

Bạn được dùng AI theo quy trình 5 bước; hãy ghi lại prompt chính và cách kiểm chứng.

### Tự làm 1 · Hồ sơ quận thứ hai

Lặp lại Bước 2–7 cho một quận bạn tự chọn (gợi ý: Providencia hoặc Estación Central) và
viết 3 câu so sánh nó với Ñuñoa. Viết code sao cho **chỉ cần đổi một biến `TEN_QUAN`**
là chạy được cho quận bất kỳ; biến này đóng vai trò cấu hình.

### Tự làm 2 · Quận có giá thấp và nhiều lựa chọn

Dùng `df.groupby("neighbourhood")["price"]` với `.agg(["median", "size"])` (xem trước
bài 5): tìm quận có **≥ 300 chỗ ở** và giá trung vị **thấp nhất**. Kiểm chứng lại con số
bằng cách lọc thủ công quận đó và gọi `.median()`.

### Tự làm 2 · AI là đề bài — vì sao bảng không đổi?

Một lời giải do AI sinh ra cho Bước 5:

```python
bao_gia = nu[["id", "room_type", "price"]].copy()
bao_gia["price"][bao_gia["room_type"] == "Entire home/apt"] *= 0.9
```

Chạy thử, đọc thông báo pandas đưa ra, giải thích bằng 2–3 câu vì sao giá không đổi, rồi viết lại đúng bằng `.loc`. Ghi lại lời giải thích vào ô dưới cùng với phần khai báo AI.


In [ ]:
# Bài tự làm 1
# Bước 2
neibourhood_name = "Providencia"
prov = df[df["neighbourhood"] == neibourhood_name]
ty_trong = len(nu) / len(df)
print(f"{neibourhood_name}: {len(prov):,} chỗ ở = {ty_trong:.1%} thị trường.")

# Bước 3
so_can = ((prov["room_type"] == "Entire home/apt") & (prov["price"] <= 60000)).sum()
print(f"{so_can} căn — phân khúc khá đông.")

# Bước 4
re_nhat = prov.nsmallest(5, "price")[["name", "price"]]
print(re_nhat)
print(f"            Giá 5 căn rẻ nhất của quận Providencia")

# Bước 5
bao_gia = nu[["id", "room_type", "price"]].copy()
bao_gia["tien_3_dem"] = bao_gia["price"] * 3
mask_nc_nu = bao_gia["room_type"] == "Entire home/apt"
bao_gia.loc[mask_nc_nu, "price"] *= 0.9
print()
print(bao_gia[["room_type", "tien_3_dem"]].head())
print("Giá đã sửa trên bản sao; prov giữ nguyên.")
print()
bao_gia["tien_3_dem"] = bao_gia["price"] * 3
print(bao_gia[["room_type", "tien_3_dem"]].head())
print(f"Đã tính lại — trung vị báo giá 3 đêm: {round(bao_gia["tien_3_dem"].median(), 1)}.")

# Bước 6
med_tp = df["price"].median()
ty_le_tho = (prov["price"] < med_tp).mean()
ty_le_sach = (prov.loc[prov["price"].notna(), "price"] < med_tp).mean()
print()
print(f"Thô: {ty_le_tho:.1%} — Sạch: {ty_le_sach:.1%}. Mẫu số đổi thì tỷ lệ đổi: lệch {(abs(ty_le_tho - ty_le_sach)):.1} điểm %.")

# Bước 7
co_cau_prov = prov["room_type"].value_counts(normalize=True)
thieu_prov = round(prov["price"].isna().mean() * 100, 2)
print(f"Nguyên căn {co_cau_nu['Entire home/apt']:.1%} · thiếu giá {thieu_prov}%")

Providencia: 2,997 chỗ ở = 9.8% thị trường.
372 căn — phân khúc khá đông.
                                                    name   price
10226                               Providencia Vibes II  1352.0
10168    La mejor ubicación en el corazón de Providencia  3068.0
16575                         cálida habitación amoblada  4446.0
16601  Arriendo mujer adulta dormitorio con baño privado  5043.0
5337                  Increible Departamento Providencia  7093.0
            Giá 5 căn rẻ nhất của quận Providencia

          room_type  tien_3_dem
0      Private room    136941.0
19  Entire home/apt    171231.0
29  Entire home/apt    180732.0
37     Private room    133314.0
38  Entire home/apt    277992.0
Giá đã sửa trên bản sao; prov giữ nguyên.

          room_type  tien_3_dem
0      Private room    136941.0
19  Entire home/apt    154107.9
29  Entire home/apt    162658.8
37     Private room    133314.0
38  Entire home/apt    250192.8
Đã tính lại — trung vị báo giá 3 đêm: 165150.9.

Thô: 29.

In [ ]:
# Bài tự làm 2
ss2 = df.groupby("neighbourhood")["price"].agg(["median", "size"])
quan_tren_300 = ss2[ss2["size"] >= 300]
quan_re_nhat = quan_tren_300.sort_values("median").head(1)
print(f"Quận thoả mãn điều kiện là: {quan_re_nhat}")


check_thu_cong = df[df["neighbourhood"] == "Estación Central"]["price"].median()
print(f"Giá trung vị của thủ công là : {check_thu_cong}")

Quận thoả mãn điều kiện là:                    median  size
neighbourhood                  
Estación Central  39941.0   440
Giá trung vị của thủ công là : 39941.0


In [ ]:
# Bài tự làm 2
bao_gia = nu[["id", "room_type", "price"]].copy()
bao_gia["price"][bao_gia["room_type"] == "Entire home/apt"] *= 0.9
# lý do giá ko đổi ở đây là do cơ chế CoW trong những phiên bản pandas mới, trong đoạn code này thì pandas đang hiểu theo 2 bước, bước thứ nhất là đang cố lọc
# theo phòng, bước thứ 2 là đang thay đổi giá của các phòng đó. Ngay tại bước 2 này thì do bạn đang cố gắng thay đổi giá trị 1 phần dữ liệu mới được trích xuất
# nên sẽ lập tức kích hoạt cơ chế CoW - lưu phần dữ liệu thay đổi đó vào một vùng nhớ mới trên RAM và giá trị gốc ko thay đổi chút nào

# Cách sửa
bao_gia.loc[bao_gia["room_type"] == "Entire home/apt", "price"] *= 0.9

/tmp/ipykernel_4575/4243578208.py:3: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  bao_gia["price"][bao_gia["room_type"] == "Entire home/apt"] *= 0.9
/tmp/ipykernel_4575/4243578208.py:3: SettingWithCopyWarning: 
A value is trying to be set o

###Không dùng AI cho cả 3 bài tự làm

## Tóm tắt bài lab

| Nội dung chính | Sẽ gặp lại ở |
|---|---|
| Thói quen 5 bước + đọc số liệu từ output | kiểm tra dataset mới và code do AI tạo |
| `df[mask]` vs `df.loc[mask, cột]`, lọc nhiều điều kiện, `nsmallest` | mọi bước lọc/chọn từ Bài 5 trở đi |
| Sửa bảng bằng `.loc`, tính lại cột phụ thuộc, bẫy hai lần ngoặc | làm sạch dữ liệu (Bài 10), pipeline BTL |
| NaN làm lệch mean-trên-bool (0.444 vs 0.467) | làm sạch dữ liệu (bài 10) |
| Hồ sơ quận → CSV, code đổi 1 biến chạy quận khác | xây dựng phân tích có thể tái sử dụng |

Bài giảng tiếp theo: index, `groupby` và ghép bảng để tổng hợp nhiều quận.